# Carvana Bad Buy Classification with Tree Ensembles

This notebook builds a tree-based machine learning pipeline for the Kaggle **Carvana: Don't Get Kicked!** dataset.

The objective is to predict whether a vehicle purchased at auction is a bad buy (`IsBadBuy`). The project covers strict time-based validation, leakage-safe preprocessing, custom implementations of tree algorithms, ensemble methods, gradient boosting, and comparison against production-grade GBDT libraries.


## 1. Imports and Global Settings

In [2]:
import os
import zipfile
import subprocess
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier as SkDecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier as SkRandomForestClassifier
from sklearn.ensemble import ExtraTreesClassifier as SkExtraTreesClassifier

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

pd.set_option("display.max_columns", 100)


## 2. Data Loading and Temporal Split

The competition data is expected to contain `training.csv`. 

The dataset is sorted by `PurchDate`, and the split is created in chronological order:

```text
train.PurchDate < validation.PurchDate < test.PurchDate
```

This design avoids using future observations to evaluate earlier periods and better reflects a real business forecasting setup.


In [3]:
def find_training_csv():
    candidates = [
        Path("data/training.csv"),
    ]
    for path in candidates:
        if path.exists():
            return path
    return None


data_path = find_training_csv()
if data_path is None:
    raise FileNotFoundError("training.csv was not found. Put it in ./data .")

print("Using data file:", data_path)


FileNotFoundError: training.csv was not found. Put it in ./data .

In [ ]:
df = pd.read_csv(data_path)
df["PurchDate"] = pd.to_datetime(df["PurchDate"])
df = df.sort_values("PurchDate").reset_index(drop=True)

TARGET = "IsBadBuy"
DATE_COL = "PurchDate"
ID_COL = "RefId" if "RefId" in df.columns else None

print(df.shape)
df.head()


(72983, 34)


,RefId,IsBadBuy,PurchDate,Auction,VehYear,VehicleAge,Make,Model,Trim,SubModel,Color,Transmission,WheelTypeID,WheelType,VehOdo,Nationality,Size,TopThreeAmericanName,MMRAcquisitionAuctionAveragePrice,MMRAcquisitionAuctionCleanPrice,MMRAcquisitionRetailAveragePrice,MMRAcquisitonRetailCleanPrice,MMRCurrentAuctionAveragePrice,MMRCurrentAuctionCleanPrice,MMRCurrentRetailAveragePrice,MMRCurrentRetailCleanPrice,PRIMEUNIT,AUCGUART,BYRNO,VNZIP1,VNST,VehBCost,IsOnlineSale,WarrantyCost
0,32389,0,2009-01-05,MANHEIM,2007,2,CHRYSLER,PACIFICA FWD 3.8L V6,Bas,4D SPORT,BLUE,AUTO,2.0,Covers,78541,AMERICAN,CROSSOVER,CHRYSLER,7261.0,8857.0,8342.0,10066.0,8709.0,10331.0,9906.0,11657.0,NaN,NaN,3453,80022,CO,6770.0,0,1389
1,32406,0,2009-01-05,MANHEIM,2005,4,FORD,FREESTAR FWD V6 3.9L,SES,4D PASSENGER 3.9L SES,SILVER,AUTO,1.0,Alloy,37676,AMERICAN,VAN,FORD,4409.0,5734.0,5262.0,6693.0,4908.0,5971.0,5801.0,6949.0,NaN,NaN,22916,80022,CO,6160.0,0,941
2,32407,0,2009-01-05,MANHEIM,2004,5,DODGE,STRATUS 4C 2.4L I4 M,SE,4D SEDAN SE,SILVER,AUTO,2.0,Covers,71680,AMERICAN,MEDIUM,CHRYSLER,3098.0,4061.0,3846.0,4886.0,3397.0,4272.0,4169.0,5114.0,NaN,NaN,3453,80022,CO,4250.0,0,1155
3,32408,0,2009-01-05,MANHEIM,2006,3,CHEVROLET,TRAILBLAZER EXT 4WD,LS,4D SUV 4.2L,WHITE,AUTO,1.0,Alloy,69456,AMERICAN,MEDIUM SUV,GM,8530.0,9883.0,9712.0,11174.0,9202.0,10794.0,10438.0,12158.0,NaN,NaN,22916,80022,CO,8180.0,0,1703
4,32409,0,2009-01-05,MANHEIM,2004,5,FORD,TAURUS 3.0L V6 EFI,SES,4D SEDAN SES DURATEC,GOLD,AUTO,1.0,Alloy,66530,AMERICAN,MEDIUM,FORD,3094.0,4230.0,3842.0,5068.0,3369.0,4492.0,4139.0,5351.0,NaN,NaN,22916,80022,CO,4900.0,0,825


In [ ]:
def strict_temporal_split(df, date_col="PurchDate", train_frac=0.33, valid_frac=0.33):
    # Create chronological splits while avoiding date overlap across train, validation, and test.
    df_sorted = df.sort_values(date_col).reset_index(drop=True).copy()
    dates = df_sorted[date_col]
    n = len(df_sorted)

    raw_train_end = int(n * train_frac)
    raw_valid_end = int(n * (train_frac + valid_frac))

    raw_train_end = min(max(raw_train_end, 1), n - 2)
    raw_valid_end = min(max(raw_valid_end, raw_train_end + 1), n - 1)

    # Move each boundary to avoid date overlap.
    train_boundary_date = dates.iloc[raw_train_end]
    valid_boundary_date = dates.iloc[raw_valid_end]

    train_end = dates.searchsorted(train_boundary_date, side="left")
    valid_end = dates.searchsorted(valid_boundary_date, side="left")

    # Safety fallback if a very rare case makes one split empty.
    if train_end == 0:
        train_end = dates.searchsorted(train_boundary_date, side="right")
    if valid_end <= train_end:
        valid_end = dates.searchsorted(valid_boundary_date, side="right")

    train_df = df_sorted.iloc[:train_end].copy()
    valid_df = df_sorted.iloc[train_end:valid_end].copy()
    test_df = df_sorted.iloc[valid_end:].copy()

    assert len(train_df) > 0 and len(valid_df) > 0 and len(test_df) > 0
    assert train_df[date_col].max() < valid_df[date_col].min()
    assert valid_df[date_col].max() < test_df[date_col].min()

    return train_df, valid_df, test_df


train_df, valid_df, test_df = strict_temporal_split(df, DATE_COL)

print("Train:", train_df.shape)
print("Valid:", valid_df.shape)
print("Test: ", test_df.shape)
print()
print("Date ranges:")
print("Train:", train_df[DATE_COL].min(), "-", train_df[DATE_COL].max())
print("Valid:", valid_df[DATE_COL].min(), "-", valid_df[DATE_COL].max())
print("Test: ", test_df[DATE_COL].min(), "-", test_df[DATE_COL].max())
print()
print("Split proportions:")
print(len(train_df) / len(df), len(valid_df) / len(df), len(test_df) / len(df))


Train: (24080, 34)
Valid: (23906, 34)
Test:  (24997, 34)

Date ranges:
Train: 2009-01-05 00:00:00 - 2009-09-10 00:00:00
Valid: 2009-09-11 00:00:00 - 2010-05-10 00:00:00
Test:  2010-05-11 00:00:00 - 2010-12-30 00:00:00

Split proportions:
0.3299398490059329 0.3275557321568036 0.34250441883726346


## 3. Leakage-Safe Feature Preprocessing

Categorical encoding is fitted only on the training split and then applied to validation and test.


In [ ]:
def add_date_features(X, date_col="PurchDate"):
    X = X.copy()
    if date_col in X.columns:
        d = pd.to_datetime(X[date_col])
        X["purchase_year"] = d.dt.year
        X["purchase_month"] = d.dt.month
        X["purchase_day"] = d.dt.day
        X["purchase_dayofweek"] = d.dt.dayofweek
        X["purchase_dayofyear"] = d.dt.dayofyear
        X["month_sin"] = np.sin(2 * np.pi * X["purchase_month"] / 12)
        X["month_cos"] = np.cos(2 * np.pi * X["purchase_month"] / 12)
        X["dow_sin"] = np.sin(2 * np.pi * X["purchase_dayofweek"] / 7)
        X["dow_cos"] = np.cos(2 * np.pi * X["purchase_dayofweek"] / 7)
        X = X.drop(columns=[date_col])
    return X


def split_X_y(part_df):
    drop_cols = [TARGET]
    if ID_COL is not None:
        drop_cols.append(ID_COL)
    X = part_df.drop(columns=drop_cols)
    X = add_date_features(X, DATE_COL)
    y = part_df[TARGET].astype(int).values
    return X, y


Xtrain_raw, ytrain = split_X_y(train_df)
Xvalid_raw, yvalid = split_X_y(valid_df)
Xtest_raw, ytest = split_X_y(test_df)

cat_cols = Xtrain_raw.select_dtypes(include=["object", "category", "string"]).columns.tolist()
num_cols = [c for c in Xtrain_raw.columns if c not in cat_cols]

print("Categorical columns:", cat_cols)
print("Numeric columns:", len(num_cols))


Categorical columns: ['Auction', 'Make', 'Model', 'Trim', 'SubModel', 'Color', 'Transmission', 'WheelType', 'Nationality', 'Size', 'TopThreeAmericanName', 'PRIMEUNIT', 'AUCGUART', 'VNST']
Numeric columns: 26


In [ ]:
class SimpleCountEncoder:

    def __init__(self, cols, normalize=True):
        self.cols = list(cols)
        self.normalize = normalize
        self.maps_ = {}

    def fit(self, X, y=None):
        X = X.copy()
        self.maps_ = {}
        for c in self.cols:
            vc = X[c].value_counts(normalize=self.normalize, dropna=False)
            self.maps_[c] = vc.to_dict()
        return self

    def transform(self, X):
        X = X.copy()
        out = pd.DataFrame(index=X.index)
        for c in self.cols:
            out[c] = X[c].map(self.maps_[c]).fillna(0.0).astype(float)
        return out

    def fit_transform(self, X, y=None):
        return self.fit(X, y).transform(X)


class TabularPreprocessor:
    # Leakage-safe preprocessing for categorical and numeric features.

    def __init__(self, cat_cols, num_cols, prefer_count_encoder=True):
        self.cat_cols = list(cat_cols)
        self.num_cols = list(num_cols)
        self.prefer_count_encoder = prefer_count_encoder
        self.encoder = None
        self.imputer = None
        self.use_count_encoder = False
        self.feature_names_ = None

    def fit(self, X, y=None):
        X = X.copy()
        for c in self.cat_cols:
            X[c] = X[c].astype("object").where(X[c].notna(), "__MISSING__").astype(str)

        self.imputer = SimpleImputer(strategy="median")
        self.imputer.fit(X[self.num_cols])

        if self.cat_cols:
            if self.prefer_count_encoder:
                # Count/frequency encoding fitted only on train.
                # Unknown categories in valid/test become 0.0.
                self.encoder = SimpleCountEncoder(self.cat_cols, normalize=True)
                self.encoder.fit(X[self.cat_cols], y)
                self.use_count_encoder = True
            else:
                try:
                    self.encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
                except TypeError:
                    self.encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)
                self.encoder.fit(X[self.cat_cols])
                self.use_count_encoder = False
        return self

    def transform(self, X):
        X = X.copy()
        for c in self.cat_cols:
            X[c] = X[c].astype("object").where(X[c].notna(), "__MISSING__").astype(str)

        X_num = pd.DataFrame(
            self.imputer.transform(X[self.num_cols]),
            columns=self.num_cols,
            index=X.index,
        )

        if self.cat_cols:
            if self.use_count_encoder:
                X_cat = self.encoder.transform(X[self.cat_cols])
                X_cat = X_cat.reset_index(drop=True)
                X_cat.index = X.index
            else:
                cat_arr = self.encoder.transform(X[self.cat_cols])
                cat_names = self.encoder.get_feature_names_out(self.cat_cols)
                X_cat = pd.DataFrame(cat_arr, columns=cat_names, index=X.index)
            X_out = pd.concat([X_num, X_cat], axis=1)
        else:
            X_out = X_num

        X_out = X_out.astype(float)
        self.feature_names_ = X_out.columns.tolist()
        return X_out

    def fit_transform(self, X, y=None):
        return self.fit(X, y).transform(X)


preprocessor = TabularPreprocessor(cat_cols, num_cols, prefer_count_encoder=True)
Xtrain = preprocessor.fit_transform(Xtrain_raw, ytrain)
Xvalid = preprocessor.transform(Xvalid_raw)
Xtest = preprocessor.transform(Xtest_raw)

Xtrain_np = Xtrain.values
Xvalid_np = Xvalid.values
Xtest_np = Xtest.values

print(Xtrain.shape, Xvalid.shape, Xtest.shape)
Xtrain.head()


(24080, 40) (23906, 40) (24997, 40)


,VehYear,VehicleAge,WheelTypeID,VehOdo,MMRAcquisitionAuctionAveragePrice,MMRAcquisitionAuctionCleanPrice,MMRAcquisitionRetailAveragePrice,MMRAcquisitonRetailCleanPrice,MMRCurrentAuctionAveragePrice,MMRCurrentAuctionCleanPrice,MMRCurrentRetailAveragePrice,MMRCurrentRetailCleanPrice,BYRNO,VNZIP1,VehBCost,IsOnlineSale,WarrantyCost,purchase_year,purchase_month,purchase_day,purchase_dayofweek,purchase_dayofyear,month_sin,month_cos,dow_sin,dow_cos,Auction,Make,Model,Trim,SubModel,Color,Transmission,WheelType,Nationality,Size,TopThreeAmericanName,PRIMEUNIT,AUCGUART,VNST
0,2007.0,2.0,2.0,78541.0,7261.0,8857.0,8342.0,10066.0,8709.0,10331.0,9906.0,11657.0,3453.0,80022.0,6770.0,0.0,1389.0,2009.0,1.0,5.0,0.0,5.0,0.5,0.866025,0.0,1.0,0.579734,0.113206,0.004028,0.185963,0.011586,0.141113,0.971927,0.436919,0.870349,0.028032,0.332600,0.999917,0.999917,0.082807
1,2003.0,6.0,1.0,60186.0,5473.0,6843.0,6411.0,7890.0,7218.0,8289.0,8295.0,9452.0,3453.0,80022.0,5360.0,0.0,1808.0,2009.0,1.0,5.0,0.0,5.0,0.5,0.866025,0.0,1.0,0.579734,0.175457,0.000581,0.003654,0.001329,0.082973,0.971927,0.515573,0.870349,0.122384,0.190490,0.999917,0.999917,0.082807
2,2006.0,3.0,1.0,76180.0,5620.0,7239.0,6570.0,8318.0,6343.0,7892.0,7350.0,9023.0,17675.0,27542.0,5100.0,0.0,1623.0,2009.0,1.0,5.0,0.0,5.0,0.5,0.866025,0.0,1.0,0.579734,0.197093,0.013538,0.067733,0.001661,0.172841,0.971927,0.515573,0.870349,0.101620,0.332600,0.999917,0.999917,0.099045
3,2003.0,6.0,1.0,90614.0,5089.0,6767.0,5996.0,7808.0,4836.0,6514.0,5723.0,7535.0,22916.0,80022.0,5150.0,0.0,1785.0,2009.0,1.0,5.0,0.0,5.0,0.5,0.866025,0.0,1.0,0.579734,0.001993,0.000042,0.008181,0.000332,0.141113,0.971927,0.515573,0.093729,0.122384,0.129651,0.999917,0.999917,0.082807
4,2004.0,5.0,1.0,89044.0,6728.0,8592.0,7766.0,9779.0,6955.0,8327.0,8011.0,9493.0,3453.0,80022.0,5660.0,0.0,2152.0,2009.0,1.0,5.0,0.0,5.0,0.5,0.866025,0.0,1.0,0.579734,0.010673,0.000125,0.000664,0.005897,0.205814,0.971927,0.515573,0.870349,0.122384,0.347259,0.999917,0.999917,0.082807


## 4. Evaluation Metric

The main metric is **Gini score**, computed from ROC AUC:

$$
	ext{Gini} = 2 \cdot 	ext{AUC} - 1
$$

This metric is commonly used for binary ranking tasks where the model outputs probabilities rather than only class labels.


In [ ]:
def gini_score(y_true, y_score):
    return 2 * roc_auc_score(y_true, y_score) - 1


def print_gini(name, y_true, y_score):
    g = gini_score(y_true, y_score)
    print(f"{name} Gini: {g:.6f}")
    return g


## 5. Custom Tree Components

This section implements reusable building blocks for tree-based models:

- `Node` for storing split information and predictions
- `DecisionTreeClassifier` using Gini impurity
- `DecisionTreeRegressor` using standard deviation reduction
- `ExtraRandomizedTreeClassifier` with randomized split thresholds


In [ ]:
def resolve_max_features(max_features, n_features):
    if max_features is None:
        return n_features
    if isinstance(max_features, str):
        if max_features == "sqrt":
            return max(1, int(np.sqrt(n_features)))
        if max_features == "log2":
            return max(1, int(np.log2(n_features)))
        raise ValueError("max_features must be None, int, float, 'sqrt', or 'log2'")
    if isinstance(max_features, float):
        if 0 < max_features <= 1:
            return max(1, int(max_features * n_features))
        raise ValueError("float max_features must be in (0, 1]")
    if isinstance(max_features, int):
        return max(1, min(max_features, n_features))
    raise ValueError("Unsupported max_features value")


class Node:
    def __init__(self, X=None, y=None, depth=0):
        self.X = X
        self.y = y
        self.depth = depth
        self.feature_index = None
        self.threshold = None
        self.left = None
        self.right = None
        self.value = None
        self.proba = None

    def is_leaf(self):
        return self.left is None and self.right is None

    def gini_impurity(self):
        if self.y is None or len(self.y) == 0:
            return 0.0
        counts = np.bincount(self.y.astype(int), minlength=2)
        probs = counts / counts.sum()
        return 1.0 - np.sum(probs ** 2)

    def std(self):
        if self.y is None or len(self.y) == 0:
            return 0.0
        return float(np.std(self.y))


In [ ]:
class DecisionTreeClassifier:
    def __init__(self, max_depth=5, min_samples_split=20, min_samples_leaf=10,
                 max_features=None, random_state=None):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.max_features = max_features
        self.random_state = random_state
        self.rng_ = np.random.default_rng(random_state)
        self.root = None
        self.n_features_ = None
        self.classes_ = np.array([0, 1])

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=int)
        self.n_features_ = X.shape[1]
        self.classes_ = np.unique(y)
        self.root = self._build_tree(X, y, depth=0)
        return self

    def predict_proba(self, X):
        X = np.asarray(X, dtype=float)
        return np.array([self._predict_proba_one(x, self.root) for x in X])

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)

    def _make_leaf(self, X, y, depth):
        node = Node(X, y, depth)
        counts = np.bincount(y.astype(int), minlength=2)
        node.value = int(np.argmax(counts))
        node.proba = counts / counts.sum() if counts.sum() > 0 else np.array([0.5, 0.5])
        return node

    def _build_tree(self, X, y, depth):
        if (
            depth >= self.max_depth
            or len(y) < self.min_samples_split
            or len(np.unique(y)) == 1
        ):
            return self._make_leaf(X, y, depth)

        feature, threshold, score = self._find_best_split(X, y)
        if feature is None:
            return self._make_leaf(X, y, depth)

        left_mask = X[:, feature] <= threshold
        right_mask = ~left_mask

        if left_mask.sum() < self.min_samples_leaf or right_mask.sum() < self.min_samples_leaf:
            return self._make_leaf(X, y, depth)

        node = Node(X, y, depth)
        node.feature_index = feature
        node.threshold = threshold
        node.left = self._build_tree(X[left_mask], y[left_mask], depth + 1)
        node.right = self._build_tree(X[right_mask], y[right_mask], depth + 1)
        return node

    def _feature_indices_for_node(self, n_features):
        k = resolve_max_features(self.max_features, n_features)
        if k >= n_features:
            return np.arange(n_features)
        return self.rng_.choice(n_features, size=k, replace=False)

    def _find_best_split(self, X, y):
        n_samples, n_features = X.shape
        best_feature, best_threshold = None, None
        best_score = np.inf
        features = self._feature_indices_for_node(n_features)
        total_pos = y.sum()

        for feature in features:
            x = X[:, feature]
            order = np.argsort(x, kind="mergesort")
            xs = x[order]
            ys = y[order]

            diff = xs[:-1] != xs[1:]
            candidate_pos = np.where(diff)[0] + 1
            if candidate_pos.size == 0:
                continue

            candidate_pos = candidate_pos[
                (candidate_pos >= self.min_samples_leaf)
                & ((n_samples - candidate_pos) >= self.min_samples_leaf)
            ]
            if candidate_pos.size == 0:
                continue

            cum_pos = np.cumsum(ys)
            left_n = candidate_pos
            right_n = n_samples - candidate_pos
            left_pos = cum_pos[candidate_pos - 1]
            right_pos = total_pos - left_pos

            left_p = left_pos / left_n
            right_p = right_pos / right_n
            g_left = 1.0 - left_p**2 - (1.0 - left_p)**2
            g_right = 1.0 - right_p**2 - (1.0 - right_p)**2
            weighted_gini = (left_n * g_left + right_n * g_right) / n_samples

            idx = np.argmin(weighted_gini)
            score = weighted_gini[idx]
            if score < best_score:
                pos = candidate_pos[idx]
                best_score = score
                best_feature = feature
                best_threshold = (xs[pos - 1] + xs[pos]) / 2.0

        return best_feature, best_threshold, best_score

    def _predict_proba_one(self, x, node):
        if node.is_leaf():
            return node.proba
        if x[node.feature_index] <= node.threshold:
            return self._predict_proba_one(x, node.left)
        return self._predict_proba_one(x, node.right)


In [ ]:
class DecisionTreeRegressor:
    def __init__(self, max_depth=3, min_samples_split=20, min_samples_leaf=10,
                 max_features=None, random_state=None):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.max_features = max_features
        self.random_state = random_state
        self.rng_ = np.random.default_rng(random_state)
        self.root = None
        self.n_features_ = None

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float)
        self.n_features_ = X.shape[1]
        self.root = self._build_tree(X, y, depth=0)
        return self

    def predict(self, X):
        X = np.asarray(X, dtype=float)
        return np.array([self._predict_one(x, self.root) for x in X])

    def predict_proba(self, X):
        # API-compatible probability method.
        return self.predict(X)

    def _make_leaf(self, X, y, depth):
        node = Node(X, y, depth)
        node.value = float(np.mean(y)) if len(y) else 0.0
        return node

    def _build_tree(self, X, y, depth):
        if depth >= self.max_depth or len(y) < self.min_samples_split or np.std(y) == 0:
            return self._make_leaf(X, y, depth)

        feature, threshold, score = self._find_best_split(X, y)
        if feature is None:
            return self._make_leaf(X, y, depth)

        left_mask = X[:, feature] <= threshold
        right_mask = ~left_mask
        if left_mask.sum() < self.min_samples_leaf or right_mask.sum() < self.min_samples_leaf:
            return self._make_leaf(X, y, depth)

        node = Node(X, y, depth)
        node.feature_index = feature
        node.threshold = threshold
        node.left = self._build_tree(X[left_mask], y[left_mask], depth + 1)
        node.right = self._build_tree(X[right_mask], y[right_mask], depth + 1)
        return node

    def _feature_indices_for_node(self, n_features):
        k = resolve_max_features(self.max_features, n_features)
        if k >= n_features:
            return np.arange(n_features)
        return self.rng_.choice(n_features, size=k, replace=False)

    def _find_best_split(self, X, y):
        n_samples, n_features = X.shape
        best_feature, best_threshold = None, None
        best_score = np.inf
        features = self._feature_indices_for_node(n_features)

        for feature in features:
            x = X[:, feature]
            order = np.argsort(x, kind="mergesort")
            xs = x[order]
            ys = y[order]

            diff = xs[:-1] != xs[1:]
            candidate_pos = np.where(diff)[0] + 1
            candidate_pos = candidate_pos[
                (candidate_pos >= self.min_samples_leaf)
                & ((n_samples - candidate_pos) >= self.min_samples_leaf)
            ]
            if candidate_pos.size == 0:
                continue

            cum_sum = np.cumsum(ys)
            cum_sq_sum = np.cumsum(ys ** 2)
            total_sum = cum_sum[-1]
            total_sq_sum = cum_sq_sum[-1]

            left_n = candidate_pos
            right_n = n_samples - candidate_pos
            left_sum = cum_sum[candidate_pos - 1]
            left_sq_sum = cum_sq_sum[candidate_pos - 1]
            right_sum = total_sum - left_sum
            right_sq_sum = total_sq_sum - left_sq_sum

            left_sse = left_sq_sum - (left_sum ** 2) / left_n
            right_sse = right_sq_sum - (right_sum ** 2) / right_n
            mse_score = (left_sse + right_sse) / n_samples

            idx = np.argmin(mse_score)
            score = mse_score[idx]
            if score < best_score:
                pos = candidate_pos[idx]
                best_score = score
                best_feature = feature
                best_threshold = (xs[pos - 1] + xs[pos]) / 2.0

        return best_feature, best_threshold, best_score

    def _predict_one(self, x, node):
        if node.is_leaf():
            return node.value
        if x[node.feature_index] <= node.threshold:
            return self._predict_one(x, node.left)
        return self._predict_one(x, node.right)


In [ ]:
class ExtraRandomizedTreeClassifier(DecisionTreeClassifier):
    # Extra randomized tree with randomized split thresholds.

    def _find_best_split(self, X, y):
        n_samples, n_features = X.shape
        best_feature, best_threshold = None, None
        best_score = np.inf
        features = self._feature_indices_for_node(n_features)

        for feature in features:
            x = X[:, feature]
            x_min, x_max = np.min(x), np.max(x)
            if x_min == x_max:
                continue

            threshold = self.rng_.uniform(x_min, x_max)
            left_mask = x <= threshold
            right_mask = ~left_mask

            n_left = left_mask.sum()
            n_right = right_mask.sum()
            if n_left < self.min_samples_leaf or n_right < self.min_samples_leaf:
                continue

            y_left = y[left_mask]
            y_right = y[right_mask]

            def gini(y_part):
                counts = np.bincount(y_part.astype(int), minlength=2)
                p = counts / counts.sum()
                return 1.0 - np.sum(p ** 2)

            score = (n_left * gini(y_left) + n_right * gini(y_right)) / n_samples
            if score < best_score:
                best_score = score
                best_feature = feature
                best_threshold = threshold

        return best_feature, best_threshold, best_score


## 6. Custom Decision Tree Validation

The custom decision tree is evaluated on the validation split using Gini score.

The target threshold for this baseline is a validation Gini score of at least `0.10`.


In [ ]:
my_tree = DecisionTreeClassifier(
    max_depth=7,
    min_samples_split=50,
    min_samples_leaf=25,
    max_features=None,
    random_state=RANDOM_STATE,
)
my_tree.fit(Xtrain_np, ytrain)

my_tree_valid_proba = my_tree.predict_proba(Xvalid_np)[:, 1]
my_tree_gini = print_gini("Custom DecisionTree valid", yvalid, my_tree_valid_proba)

print("Requirement >= 0.10:", "PASS" if my_tree_gini >= 0.10 else "FAIL")


Custom DecisionTree valid Gini: 0.450891
Requirement >= 0.10: PASS


## 7. scikit-learn Decision Tree Benchmark

The custom implementation is compared with `sklearn.tree.DecisionTreeClassifier`.

The scikit-learn implementation is expected to be stronger because it is highly optimized, includes additional regularization controls, and uses efficient split-search internals.


In [ ]:
sk_tree = SkDecisionTreeClassifier(
    max_depth=7,
    min_samples_leaf=25,
    random_state=RANDOM_STATE,
)
sk_tree.fit(Xtrain, ytrain)
sk_tree_valid_proba = sk_tree.predict_proba(Xvalid)[:, 1]
sk_tree_gini = print_gini("sklearn DecisionTree valid", yvalid, sk_tree_valid_proba)

print("sklearn better than custom tree:", sk_tree_gini > my_tree_gini)


sklearn DecisionTree valid Gini: 0.419568
sklearn better than custom tree: False


## 8. Custom Random Forest

A random forest improves a single decision tree by training multiple trees on bootstrapped samples and averaging their predicted probabilities.

The ensemble is designed to reduce variance and improve validation Gini compared with a standalone tree.



This implementation uses:

- bootstrap sampling of rows,
- random feature subsets at each split,
- averaging probabilities across trees,
- fixed random seed.


In [ ]:
class RandomForestClassifier:
    def __init__(self, n_estimators=40, max_depth=7, max_features="sqrt",
                 sample_size=1.0, min_samples_split=50, min_samples_leaf=25,
                 random_state=42):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.max_features = max_features
        self.sample_size = sample_size
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.random_state = random_state
        self.rng_ = np.random.default_rng(random_state)
        self.trees = []

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=int)
        n_samples = X.shape[0]
        bag_size = max(1, int(self.sample_size * n_samples))
        self.trees = []

        for i in range(self.n_estimators):
            idx = self.rng_.integers(0, n_samples, size=bag_size)
            tree_seed = int(self.rng_.integers(0, 1_000_000_000))
            tree = DecisionTreeClassifier(
                max_depth=self.max_depth,
                min_samples_split=self.min_samples_split,
                min_samples_leaf=self.min_samples_leaf,
                max_features=self.max_features,
                random_state=tree_seed,
            )
            tree.fit(X[idx], y[idx])
            self.trees.append(tree)
        return self

    def predict_proba(self, X):
        X = np.asarray(X, dtype=float)
        probas = np.array([tree.predict_proba(X) for tree in self.trees])
        return probas.mean(axis=0)

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)


In [ ]:
my_rf = RandomForestClassifier(
    n_estimators=25,
    max_depth=8,
    max_features="sqrt",
    sample_size=1.0,
    min_samples_split=60,
    min_samples_leaf=25,
    random_state=RANDOM_STATE,
)
my_rf.fit(Xtrain_np, ytrain)

my_rf_valid_proba = my_rf.predict_proba(Xvalid_np)[:, 1]
my_rf_gini = print_gini("Custom RandomForest valid", yvalid, my_rf_valid_proba)

print("Requirement >= 0.15:", "PASS" if my_rf_gini >= 0.15 else "FAIL")
print("Improves custom single tree:", my_rf_gini > my_tree_gini)


Custom RandomForest valid Gini: 0.458153
Requirement >= 0.15: PASS
Improves custom single tree: True


## 9. Gradient Boosted Decision Trees from Scratch

This section implements a simplified binary GBDT classifier.

Each new tree is trained on the negative gradient of the binary cross-entropy loss. The model updates raw logits incrementally and converts them to probabilities using the sigmoid function.


In [ ]:
sk_rf = SkRandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    min_samples_leaf=25,
    max_features="sqrt",
    n_jobs=-1,
    random_state=RANDOM_STATE,
)
sk_rf.fit(Xtrain, ytrain)
sk_rf_valid_proba = sk_rf.predict_proba(Xvalid)[:, 1]
sk_rf_gini = print_gini("sklearn RandomForest valid", yvalid, sk_rf_valid_proba)


sklearn RandomForest valid Gini: 0.471492


## 10. Custom GBDT Validation

The custom GBDT model is trained and evaluated on the validation split.

This provides a transparent implementation of the boosting idea before comparing against specialized libraries.


In [ ]:
class GBDTClassifier:
    def __init__(self, max_depth=3, number_of_trees=80, max_features="sqrt",
                 learning_rate=0.08, min_samples_split=50, min_samples_leaf=25,
                 random_state=42):
        self.max_depth = max_depth
        self.number_of_trees = number_of_trees
        self.max_features = max_features
        self.learning_rate = learning_rate
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.random_state = random_state
        self.rng_ = np.random.default_rng(random_state)
        self.trees = []
        self.init_raw_ = None

    @staticmethod
    def _sigmoid(x):
        return 1.0 / (1.0 + np.exp(-np.clip(x, -50, 50)))

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float)

        eps = 1e-6
        p0 = np.clip(y.mean(), eps, 1 - eps)
        self.init_raw_ = np.log(p0 / (1 - p0))
        raw_pred = np.full(len(y), self.init_raw_, dtype=float)
        self.trees = []

        for _ in range(self.number_of_trees):
            prob = self._sigmoid(raw_pred)
            negative_gradient = y - prob

            tree_seed = int(self.rng_.integers(0, 1_000_000_000))
            tree = DecisionTreeRegressor(
                max_depth=self.max_depth,
                min_samples_split=self.min_samples_split,
                min_samples_leaf=self.min_samples_leaf,
                max_features=self.max_features,
                random_state=tree_seed,
            )
            tree.fit(X, negative_gradient)
            update = tree.predict(X)
            raw_pred += self.learning_rate * update
            self.trees.append(tree)
        return self

    def predict_proba(self, X):
        X = np.asarray(X, dtype=float)
        raw_pred = np.full(X.shape[0], self.init_raw_, dtype=float)
        for tree in self.trees:
            raw_pred += self.learning_rate * tree.predict(X)
        prob = self._sigmoid(raw_pred)
        return np.column_stack([1 - prob, prob])

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)


In [ ]:
my_gbdt = GBDTClassifier(
    max_depth=3,
    number_of_trees=60,
    max_features="sqrt",
    learning_rate=0.08,
    min_samples_split=60,
    min_samples_leaf=25,
    random_state=RANDOM_STATE,
)
my_gbdt.fit(Xtrain_np, ytrain)

my_gbdt_valid_proba = my_gbdt.predict_proba(Xvalid_np)[:, 1]
my_gbdt_gini = print_gini("Custom GBDT valid", yvalid, my_gbdt_valid_proba)


Custom GBDT valid Gini: 0.462923


## 11. LightGBM, CatBoost, and XGBoost

This section compares the custom models with widely used gradient boosting libraries.

Key differences:

- **LightGBM** uses histogram-based split search and leaf-wise tree growth, making it fast and effective on tabular data.
- **CatBoost** handles categorical variables natively using ordered target statistics designed to reduce target leakage.
- **XGBoost** is highly configurable and strongly regularized. Its **DART** booster randomly drops trees during boosting, similar in spirit to dropout, which can reduce overfitting.



In [ ]:
def prepare_raw_tree_data(X_train_raw, X_valid_raw, X_test_raw, cat_cols, num_cols):
    Xtr = X_train_raw.copy()
    Xva = X_valid_raw.copy()
    Xte = X_test_raw.copy()

    for c in cat_cols:
        for X in (Xtr, Xva, Xte):
            X[c] = X[c].astype("object").where(X[c].notna(), "__MISSING__").astype(str)

    medians = Xtr[num_cols].median(numeric_only=True)
    for X in (Xtr, Xva, Xte):
        X[num_cols] = X[num_cols].fillna(medians)

    return Xtr, Xva, Xte


Xtrain_raw_model, Xvalid_raw_model, Xtest_raw_model = prepare_raw_tree_data(
    Xtrain_raw, Xvalid_raw, Xtest_raw, cat_cols, num_cols
)


In [ ]:
model_results = []
model_registry = {}


def add_result(name, model, valid_proba, data_kind):
    g = gini_score(yvalid, valid_proba)
    model_results.append({"model": name, "valid_gini": g, "data_kind": data_kind})
    model_registry[name] = {"model": model, "data_kind": data_kind}
    print(f"{name} valid Gini: {g:.6f}")

# Add already trained models.
add_result("Custom DecisionTree", my_tree, my_tree_valid_proba, "encoded_np")
add_result("sklearn DecisionTree", sk_tree, sk_tree_valid_proba, "encoded_df")
add_result("Custom RandomForest", my_rf, my_rf_valid_proba, "encoded_np")
add_result("sklearn RandomForest", sk_rf, sk_rf_valid_proba, "encoded_df")
add_result("Custom GBDT", my_gbdt, my_gbdt_valid_proba, "encoded_np")


Custom DecisionTree valid Gini: 0.450891
sklearn DecisionTree valid Gini: 0.419568
Custom RandomForest valid Gini: 0.458153
sklearn RandomForest valid Gini: 0.471492
Custom GBDT valid Gini: 0.462923


In [ ]:
# Install optional GBDT libraries if they are not already available.
%pip install lightgbm catboost xgboost -q

In [ ]:
# LightGBM model
try:
    import lightgbm as lgb
    from lightgbm import LGBMClassifier

    lgbm = LGBMClassifier(
        n_estimators=200,
        learning_rate=0.03,
        num_leaves=31,
        max_depth=-1,
        min_child_samples=50,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        objective="binary",
        verbose=-1,
        force_col_wise=True,
        random_state=RANDOM_STATE,
        n_jobs=1,
    )

    lgbm.fit(
        Xtrain,
        ytrain,
        eval_set=[(Xvalid, yvalid)],
        eval_metric="auc",
        callbacks=[lgb.early_stopping(80, verbose=False), lgb.log_evaluation(0)],
    )
    lgbm_valid_proba = lgbm.predict_proba(Xvalid)[:, 1]
    add_result("LightGBM", lgbm, lgbm_valid_proba, "encoded_df")
except Exception as exc:
    print("LightGBM skipped:", exc)


LightGBM valid Gini: 0.481317


In [ ]:
# CatBoost model with native categorical features
try:
    from catboost import CatBoostClassifier

    cat_model = CatBoostClassifier(
        iterations=150,
        learning_rate=0.03,
        depth=5,
        l2_leaf_reg=8,
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=RANDOM_STATE,
        verbose=False,
        allow_writing_files=False,
        thread_count=1,
    )

    cat_model.fit(
        Xtrain_raw_model,
        ytrain,
        cat_features=cat_cols,
        eval_set=(Xvalid_raw_model, yvalid),
        use_best_model=True,
        early_stopping_rounds=80,
    )
    cat_valid_proba = cat_model.predict_proba(Xvalid_raw_model)[:, 1]
    add_result("CatBoost", cat_model, cat_valid_proba, "raw_cat")
except Exception as exc:
    print("CatBoost skipped:", exc)


CatBoost valid Gini: 0.480479


In [ ]:
# XGBoost model with DART booster
try:
    from xgboost import XGBClassifier

    xgb_dart = XGBClassifier(
        n_estimators=150,
        max_depth=4,
        learning_rate=0.03,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_alpha=0.05,
        reg_lambda=2.0,
        objective="binary:logistic",
        eval_metric="auc",
        booster="dart",
        rate_drop=0.05,
        skip_drop=0.5,
        tree_method="hist",
        random_state=RANDOM_STATE,
        n_jobs=1,
    )

    try:
        xgb_dart.fit(Xtrain, ytrain, eval_set=[(Xvalid, yvalid)], verbose=False)
    except TypeError:
        xgb_dart.fit(Xtrain, ytrain)

    xgb_valid_proba = xgb_dart.predict_proba(Xvalid)[:, 1]
    add_result("XGBoost_DART", xgb_dart, xgb_valid_proba, "encoded_df")
except Exception as exc:
    print("XGBoost skipped:", exc)


XGBoost_DART valid Gini: 0.479822


In [ ]:
results_df = pd.DataFrame(model_results).sort_values("valid_gini", ascending=False).reset_index(drop=True)
results_df


,model,valid_gini,data_kind
0,LightGBM,0.481317,encoded_df
1,CatBoost,0.480479,raw_cat
2,XGBoost_DART,0.479822,encoded_df
3,sklearn RandomForest,0.471492,encoded_df
4,Custom GBDT,0.462923,encoded_np
5,Custom RandomForest,0.458153,encoded_np
6,Custom DecisionTree,0.450891,encoded_np
7,sklearn DecisionTree,0.419568,encoded_df


## 12. Final Test Evaluation

The best model is selected using validation Gini only. After selection, the model is evaluated on train, validation, and test splits.

This final comparison helps identify whether the selected model generalizes to a later time period or overfits the training data.


In [ ]:
def predict_model_proba(entry, split):
    model = entry["model"]
    data_kind = entry["data_kind"]

    if data_kind == "encoded_np":
        X_map = {"train": Xtrain_np, "valid": Xvalid_np, "test": Xtest_np}
    elif data_kind == "encoded_df":
        X_map = {"train": Xtrain, "valid": Xvalid, "test": Xtest}
    elif data_kind == "raw_cat":
        X_map = {"train": Xtrain_raw_model, "valid": Xvalid_raw_model, "test": Xtest_raw_model}
    else:
        raise ValueError(f"Unknown data_kind: {data_kind}")

    return model.predict_proba(X_map[split])[:, 1]


best_name = results_df.iloc[0]["model"]
best_entry = model_registry[best_name]

train_proba = predict_model_proba(best_entry, "train")
valid_proba = predict_model_proba(best_entry, "valid")
test_proba = predict_model_proba(best_entry, "test")

final_scores = pd.DataFrame({
    "split": ["train", "valid", "test"],
    "gini": [
        gini_score(ytrain, train_proba),
        gini_score(yvalid, valid_proba),
        gini_score(ytest, test_proba),
    ],
})

print("Best model selected by validation Gini:", best_name)
display(final_scores)

valid_test_drop = final_scores.loc[final_scores["split"] == "valid", "gini"].iloc[0] - final_scores.loc[final_scores["split"] == "test", "gini"].iloc[0]
train_valid_gap = final_scores.loc[final_scores["split"] == "train", "gini"].iloc[0] - final_scores.loc[final_scores["split"] == "valid", "gini"].iloc[0]

print(f"Valid - Test Gini drop: {valid_test_drop:.6f}")
print(f"Train - Valid Gini gap: {train_valid_gap:.6f}")



Best model selected by validation Gini: LightGBM


,split,gini
0,train,0.607713
1,valid,0.481317
2,test,0.500318


Valid - Test Gini drop: -0.019001
Train - Valid Gini gap: 0.126396


## 13. Final Model Interpretation

The best model selected by validation Gini is **LightGBM**.

It obtains a Gini score of approximately `0.6077` on the training split, `0.4813` on validation, and `0.5003` on test.

The test score is slightly higher than the validation score, so there is no degradation from validation to test. This suggests that the model generalizes well to the later time period.

At the same time, the model shows a moderate train-validation gap, which is expected for a high-capacity gradient boosting model. However, the test Gini remains stable and slightly higher than the validation Gini, suggesting that the model generalizes reasonably well to the later time period.
